# Olist E-Commerce Sales Analysis
**Author:** Noorjahat Afreen  
**Internship:** Syntecxhub Internship Program

## Project overview
This notebook analyzes historical Olist Brazilian e-commerce data. It explores item sales, order volume, regional performance, customer review scores, and delivery performance where the relevant columns are available.

**Dataset source:** [Olist Brazilian E-Commerce Dataset on Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

> Note: Item sales are the sum of product item prices, not company profit. Results depend on the dataset files provided when running this notebook. This notebook does not contain private or live company data.


## 1. Import libraries
Run the cells from top to bottom in Google Colab or Jupyter Notebook.

In [ ]:
import os
import glob
import warnings
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

OUTPUT_DIR = "olist_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries loaded successfully.")

## 2. Load the data
Upload your merged `Olist_Sales_Analysis.csv` file when prompted. If you already have it in the current folder, the notebook will use it automatically.

In [ ]:
from pathlib import Path

candidate_files = [
    Path("Olist_Sales_Analysis.csv"),
    Path("/content/Olist_Sales_Analysis.csv"),
    Path("/content/olist_company_data/Olist_Sales_Analysis.csv")
]

DATA_FILE = next((p for p in candidate_files if p.exists()), None)

if DATA_FILE is None:
    try:
        from google.colab import files
        print("Upload Olist_Sales_Analysis.csv")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file uploaded.")
        DATA_FILE = Path(csv_files[0])
    except ImportError:
        raise FileNotFoundError(
            "Place Olist_Sales_Analysis.csv in the same folder as this notebook, then run this cell again."
        )

df = pd.read_csv(DATA_FILE, low_memory=False)
print("Loaded:", DATA_FILE)
print("Rows:", f"{len(df):,}", "| Columns:", len(df.columns))
display(df.head())
print("\nColumn names:")
print(df.columns.tolist())

## 3. Clean and inspect the data

In [ ]:
# Standardize column names for easier analysis
df.columns = [str(c).strip() for c in df.columns]

print("Dataset shape:", df.shape)
print("\nMissing values (top 20):")
display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing_values"))

print("\nDuplicate rows:", int(df.duplicated().sum()))

## 4. Identify important columns
The merged file may use slightly different column names. This cell detects common Olist column names automatically.

In [ ]:
def find_col(possible_names, columns=None):
    columns = list(df.columns if columns is None else columns)
    normalized = {str(c).lower().strip(): c for c in columns}
    for name in possible_names:
        if name.lower() in normalized:
            return normalized[name.lower()]
    return None

order_col = find_col(["order_id", "orderid"])
price_col = find_col(["price", "item_price", "total_item_sales"])
product_col = find_col(["product_id", "productid"])
state_col = find_col(["customer_state", "state"])
review_col = find_col(["review_score", "score"])
actual_delivery_col = find_col(["order_delivered_customer_date", "actual_delivery_date", "delivered_date"])
estimated_delivery_col = find_col(["order_estimated_delivery_date", "estimated_delivery_date"])
order_date_col = find_col(["order_purchase_timestamp", "order_date", "purchase_date"])
category_col = find_col(["product_category_name", "category", "product_category"])

detected = {
    "Order ID": order_col,
    "Item price": price_col,
    "Product ID": product_col,
    "Customer state": state_col,
    "Review score": review_col,
    "Actual delivery date": actual_delivery_col,
    "Estimated delivery date": estimated_delivery_col,
    "Order date": order_date_col,
    "Product category": category_col,
}
display(pd.DataFrame({"Detected column": detected}))


## 5. Sales and order KPIs

In [ ]:
total_rows = len(df)
total_orders = df[order_col].nunique() if order_col else None
total_items = total_rows
total_sales = pd.to_numeric(df[price_col], errors="coerce").sum() if price_col else None
unique_products = df[product_col].nunique() if product_col else None

kpis = pd.DataFrame([
    ["Rows in analysis file", total_rows],
    ["Unique orders", total_orders],
    ["Items/records in file", total_items],
    ["Sum of item prices (R$ if original Olist prices)", total_sales],
    ["Unique products", unique_products],
], columns=["Metric", "Value"])

display(kpis)
kpis.to_csv(os.path.join(OUTPUT_DIR, "Olist_Project_KPI_Summary.csv"), index=False)

if price_col:
    print(f"Total item-price sum: R$ {total_sales:,.2f}")
    print("Important: this is item-price sales, not profit or necessarily total company revenue.")
else:
    print("No item-price column was detected; sales KPIs cannot be calculated from this file.")

## 6. Sales by customer state

In [ ]:
if state_col and price_col:
    state_sales = (
        df.assign(_price=pd.to_numeric(df[price_col], errors="coerce"))
          .groupby(state_col, dropna=False)["_price"]
          .sum()
          .sort_values(ascending=False)
          .rename("item_sales")
          .reset_index()
    )
    display(state_sales.head(10))
    state_sales.to_csv(os.path.join(OUTPUT_DIR, "Olist_State_Sales.csv"), index=False)

    top_states = state_sales.dropna().head(10).sort_values("item_sales")
    plt.figure(figsize=(10, 6))
    plt.barh(top_states[state_col].astype(str), top_states["item_sales"])
    plt.title("Top 10 States by Item Sales")
    plt.xlabel("Sum of item prices")
    plt.ylabel("Customer state")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Top_States_by_Sales.png"), dpi=160)
    plt.show()
else:
    print("State sales chart skipped: state and/or item-price column not found.")

## 7. Monthly sales trend

In [ ]:
if order_date_col and price_col:
    trend = df[[order_date_col, price_col]].copy()
    trend[order_date_col] = pd.to_datetime(trend[order_date_col], errors="coerce")
    trend[price_col] = pd.to_numeric(trend[price_col], errors="coerce")
    trend = trend.dropna(subset=[order_date_col, price_col])
    trend["month"] = trend[order_date_col].dt.to_period("M").astype(str)
    monthly_sales = trend.groupby("month", as_index=False)[price_col].sum()
    monthly_sales = monthly_sales.rename(columns={price_col: "item_sales"})
    display(monthly_sales.tail(12))
    monthly_sales.to_csv(os.path.join(OUTPUT_DIR, "Olist_Monthly_Sales.csv"), index=False)

    plt.figure(figsize=(12, 5))
    plt.plot(monthly_sales["month"], monthly_sales["item_sales"], marker="o")
    plt.title("Monthly Item Sales Trend")
    plt.xlabel("Month")
    plt.ylabel("Sum of item prices")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Monthly_Sales_Trend.png"), dpi=160)
    plt.show()
else:
    print("Monthly trend skipped: order-date and/or item-price column not found.")

## 8. Product category analysis

In [ ]:
if category_col and price_col:
    category_sales = (
        df.assign(_price=pd.to_numeric(df[price_col], errors="coerce"))
          .groupby(category_col, dropna=False)["_price"]
          .sum()
          .sort_values(ascending=False)
          .rename("item_sales")
          .reset_index()
    )
    display(category_sales.head(10))
    category_sales.to_csv(os.path.join(OUTPUT_DIR, "Olist_Category_Sales.csv"), index=False)

    top_categories = category_sales.dropna().head(10).sort_values("item_sales")
    plt.figure(figsize=(10, 6))
    plt.barh(top_categories[category_col].astype(str), top_categories["item_sales"])
    plt.title("Top 10 Product Categories by Item Sales")
    plt.xlabel("Sum of item prices")
    plt.ylabel("Product category")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Top_Categories_by_Sales.png"), dpi=160)
    plt.show()
else:
    print("Category analysis skipped: category and/or item-price column not found.")

## 9. Customer review analysis

In [ ]:
if review_col:
    reviews = pd.to_numeric(df[review_col], errors="coerce").dropna()
    review_counts = reviews.value_counts().sort_index().rename_axis("review_score").reset_index(name="count")
    display(review_counts)
    review_counts.to_csv(os.path.join(OUTPUT_DIR, "Olist_Review_Analysis.csv"), index=False)

    plt.figure(figsize=(8, 5))
    plt.bar(review_counts["review_score"].astype(str), review_counts["count"])
    plt.title("Customer Review Score Distribution")
    plt.xlabel("Review score")
    plt.ylabel("Number of reviews")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Review_Score_Distribution.png"), dpi=160)
    plt.show()
else:
    print("Review analysis skipped: review-score column not found. If needed, merge the reviews table into your main CSV.")

## 10. Delivery performance

In [ ]:
if actual_delivery_col and estimated_delivery_col:
    delivery = df[[actual_delivery_col, estimated_delivery_col]].copy()
    delivery[actual_delivery_col] = pd.to_datetime(delivery[actual_delivery_col], errors="coerce")
    delivery[estimated_delivery_col] = pd.to_datetime(delivery[estimated_delivery_col], errors="coerce")
    delivery = delivery.dropna(subset=[actual_delivery_col, estimated_delivery_col])
    delivery["delivery_result"] = (
        delivery[actual_delivery_col] <= delivery[estimated_delivery_col]
    ).map({True: "On Time or Early", False: "Late"})
    delivery_summary = delivery["delivery_result"].value_counts().rename_axis("delivery_result").reset_index(name="count")
    delivery_summary["percentage"] = delivery_summary["count"] / delivery_summary["count"].sum() * 100
    display(delivery_summary)
    delivery_summary.to_csv(os.path.join(OUTPUT_DIR, "Olist_Delivery_Analysis.csv"), index=False)

    plt.figure(figsize=(7, 5))
    plt.bar(delivery_summary["delivery_result"], delivery_summary["count"])
    plt.title("Delivery Performance")
    plt.xlabel("Delivery result")
    plt.ylabel("Number of orders")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Delivery_Performance.png"), dpi=160)
    plt.show()
else:
    print("Delivery analysis skipped: actual and estimated delivery date columns were not found.")

## 11. Export outputs
The analysis tables and charts are saved in the `olist_analysis_outputs` folder.

In [ ]:
print("Output files:")
for path in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*"))):
    print("-", path)

# In Google Colab, optionally download a ZIP containing generated outputs.
try:
    from google.colab import files
    import shutil
    zip_path = shutil.make_archive("Olist_Analysis_Outputs", "zip", OUTPUT_DIR)
    print("\nCreated:", zip_path)
    files.download(zip_path)
except ImportError:
    print("\nTo download outputs outside Colab, use the file browser or zip the output folder.")

## 12. Conclusion
This project demonstrates exploratory analysis of historical e-commerce data. The results can help describe item sales, customer review patterns, geographic distribution, and delivery outcomes. Findings should be interpreted in context: the dataset is historical, item-price sums are not profit, and analyses are only produced when the necessary columns are present.